In [1]:
# W7D4: Install Ollama and required system dependency

!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

print("Ollama installation completed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama installation completed.


In [2]:
# W7D4: Start Ollama server

import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for attempt in range(30):
    try:
        response = requests.get(
            "http://localhost:11434/api/tags",
            timeout=2
        )
        if response.status_code == 200:
            print("Ollama server is running!")
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not start.")

Ollama server is running!


In [3]:
# W7D4: Pull models for comparison

!ollama pull llama3.2:3b
!ollama pull qwen2.5:3b

!ollama list



NAME           ID              SIZE      MODIFIED               
qwen2.5:3b     357c53fb659c    1.9 GB    Less than a second ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    21 seconds ago            


In [4]:
# W7D4: First local LLM inference

import requests

def ask_ollama(model, prompt, system_prompt="You are a helpful AI assistant."):
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            "stream": False
        },
        timeout=180
    )
    response.raise_for_status()
    return response.json()["message"]["content"]

answer = ask_ollama(
    "llama3.2:3b",
    "Explain Retrieval-Augmented Generation in simple terms."
)

print(answer)

Retrieval-Augmented Generation (RAG) is a type of AI model that combines two powerful techniques: retrieval and generation.

**Generation**: In traditional language generation models, the AI starts with a prompt and generates text on its own. It's like writing a story from scratch.

**Retrieval**: In retrieval models, the AI looks up existing knowledge to answer a question. It's like searching a bookshelf for a specific book.

**Retrieval-Augmented Generation**: In RAG, the AI uses retrieval to look up relevant information related to the prompt, and then uses that information to generate new text. It's like writing a story that starts with a research paper you've already read.

Here's a step-by-step breakdown:

1. The model receives a prompt (e.g., "Write a story about a futuristic city").
2. The model uses retrieval to look up relevant information about cities in the future, such as futuristic architecture, transportation, and technology.
3. The model uses this information to generate

In [5]:
# W7D4: Test five prompts using Llama 3.2

system_prompt = """
You are a concise AI/ML tutor.
Give accurate, beginner-friendly answers with a short example when useful.
"""

prompts = [
    "What is a large language model?",
    "Explain embeddings in simple terms.",
    "What is a vector database?",
    "How does RAG reduce hallucinations?",
    "Explain the difference between retrieval and generation."
]

for i, prompt in enumerate(prompts, start=1):
    print(f"\n--- Prompt {i}: {prompt} ---")
    print(ask_ollama("llama3.2:3b", prompt, system_prompt))


--- Prompt 1: What is a large language model? ---
A large language model (LLM) is a type of artificial intelligence (AI) model that uses natural language processing (NLP) to understand and generate human-like language. These models are trained on massive amounts of text data, which enables them to learn patterns, relationships, and context within language.

Large language models can perform various tasks such as:

- Text generation
- Language translation
- Summarization
- Sentiment analysis

Here's a simple example of a large language model in action:

**Example:**
A language model is asked to write a short paragraph summarizing the main points of a research article about climate change. The model uses its learned patterns and relationships to generate a coherent and informative summary.

**Example Code:**
```python
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load pre-trained LLM
model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")
tokenizer = Aut

In [6]:
# W7D4: Compare Llama 3.2 and Qwen 2.5

comparison_questions = [
    "Explain RAG in three sentences.",
    "What are the benefits of vector databases?",
    "How can we evaluate an AI-generated answer?"
]

comparison_results = []

for question in comparison_questions:
    print("\n" + "=" * 70)
    print("QUESTION:", question)

    llama_answer = ask_ollama("llama3.2:3b", question)
    qwen_answer = ask_ollama("qwen2.5:3b", question)

    comparison_results.append({
        "question": question,
        "llama3.2:3b": llama_answer,
        "qwen2.5:3b": qwen_answer
    })

    print("\nLLAMA 3.2:\n", llama_answer)
    print("\nQWEN 2.5:\n", qwen_answer)


QUESTION: Explain RAG in three sentences.

LLAMA 3.2:
 RAG is an acronym that stands for "Reaching and Rebuilding for Children", a non-profit organization that provides support and care for children who have experienced trauma, abuse, or neglect. RAG aims to help these children recover and heal by providing a safe and nurturing environment, as well as accessing necessary medical, educational, and emotional support. The organization works closely with families, community partners, and healthcare professionals to ensure that each child receives comprehensive care and support tailored to their unique needs.

QWEN 2.5:
 RAG stands for Retrieval-Augmented Generation, which is a method in artificial intelligence that combines the strengths of retrieval-based and generative AI systems. It uses existing information sources, such as documents or knowledge bases, to enhance the output of generative models like language models, making the generated text more accurate, relevant, and contextually 

In [7]:
# W7D4: Save model comparison results

import json
from pathlib import Path

project_dir = Path("/content/W7D4_LlamaIndex_Ollama_Local_RAG")
project_dir.mkdir(parents=True, exist_ok=True)

(project_dir / "model_comparison.json").write_text(
    json.dumps(comparison_results, indent=2)
)

comparison_notes = """
# W7D4 — Local LLM Comparison

Models tested:
- llama3.2:3b
- qwen2.5:3b

Both models received the same three questions.

Comparison criteria:
1. Relevance to the question
2. Factual accuracy
3. Clarity and readability
4. Completeness
5. Instruction following

Review the generated responses and record observed differences.
Do not assume one model is better without evaluating its outputs.
"""

(project_dir / "comparison_notes.md").write_text(
    comparison_notes.strip() + "\n"
)

print("Saved model_comparison.json and comparison_notes.md")

Saved model_comparison.json and comparison_notes.md


In [8]:
# W7D4: Save reusable inference script and project documentation

script = '''
import requests

def ask_ollama(model, prompt, system_prompt="You are a helpful AI assistant."):
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            "stream": False
        },
        timeout=180
    )
    response.raise_for_status()
    return response.json()["message"]["content"]

if __name__ == "__main__":
    question = "Explain local LLM inference."
    print(ask_ollama("llama3.2:3b", question))
'''

readme = """
# W7D4 — LlamaIndex + Ollama: Local RAG

- Installed Ollama and pulled two local language models.
- Tested five prompts with a custom system prompt.
- Compared Llama 3.2 and Qwen 2.5 on three identical questions.
- Saved model responses and comparison notes.

## Run
Ensure Ollama is installed and running, and the selected model is available.

Install the Python dependency:
    pip install requests

Run the script:
    python ollama_inference.py
"""

(project_dir / "ollama_inference.py").write_text(script.strip() + "\n")
(project_dir / "README.md").write_text(readme.strip() + "\n")

print("Created ollama_inference.py and README.md")

Created ollama_inference.py and README.md


In [9]:
# W7D4: Download project files individually

from google.colab import files
import time

for filename in [
    "ollama_inference.py",
    "README.md",
    "comparison_notes.md",
    "model_comparison.json"
]:
    path = project_dir / filename
    print("Downloading:", filename)
    files.download(str(path))
    time.sleep(2)

Downloading: ollama_inference.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: README.md


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: model_comparison.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>